# TTS 동음이의어 장단음 개선 탐구 (Colab 실행용)

**문제**: '눈(目)'과 '눈ː(雪)', '말(馬)'과 '말ː(言)'처럼 글자는 같아도 소리의 길이가 다른 단어가 있다.
TTS는 글자만 보고 읽기 때문에 이 차이를 거의 반영하지 못한다.

**두 가지 해결 방법**
1. **입력을 바꾸는 방법**: 형태소 분석기(Kiwi)와 Gemini로 장음을 판별한 뒤, 상용 TTS 입력(SSML 또는 철자)에 반영
2. **모델 내부를 바꾸는 방법**: 사전학습 VITS 모델(`facebook/mms-tts-kor`)의 duration 텐서에 hook을 걸어 장음 모음 프레임에 가중치를 곱함

마지막에 두 방법의 음성을 멜 스펙트로그램으로 비교한다.

> 런타임은 CPU로도 충분하다. Gemini API 키는 왼쪽 🔑(보안 비밀)에 `GEMINI_API_KEY`라는 이름으로 넣는다.
> Google Cloud TTS 키(`GOOGLE_TTS_API_KEY`)는 없어도 되며, 없으면 gTTS와 철자 변형으로 대신한다.
> 저장소가 비공개라면 `git clone`이 실패한다. 그때는 GitHub에서 ZIP으로 내려받아 Colab에 올리고 압축을 푼 뒤 `%cd`로 그 폴더에 들어간다.

In [ ]:
!git clone -q -b claude/clever-lamport-uucwuo https://github.com/hatnimi/claude_repo.git
%cd claude_repo
!pip -q install -r requirements.txt
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.font_manager as fm; fm._load_fontmanager(try_read_cache=False)

In [ ]:
import os, sys, json
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
try:
    os.environ["GOOGLE_TTS_API_KEY"] = userdata.get("GOOGLE_TTS_API_KEY")
except Exception:
    print("Google Cloud TTS 키 없음 → gTTS + 철자 변형 방식으로 진행")
sys.path.insert(0, "src")
from IPython.display import Audio, Image, Markdown, display

## 1단계. 형태소 분석: 판별 대상 찾기
Kiwi는 '눈이'를 `눈/NNG + 이/JKS`로 나눠 준다. 그래서 조사가 붙어 있어도 명사 '눈'을 정확히 찾을 수 있다.
그러나 두 '눈'은 품사가 똑같이 NNG이므로 **의미는 구별하지 못한다** → 2단계가 필요한 이유.

'첫눈'은 `첫 + 눈`으로 나뉘지만, 어절 중간이라 장음이 될 수 없다(표준 발음법 제6항: 긴소리는 단어의 첫음절에서만 나타난다).

In [ ]:
from morph import get_kiwi, find_candidates
s = "올해 첫눈은 눈이 부시게 하얗게 내렸다."
print([(t.form, t.tag) for t in get_kiwi().tokenize(s)])
for c in find_candidates(s):
    print(c.form, c.start, "어절 첫머리" if c.word_initial else "어절 중간 → 장음 불가", [x["id"] for x in c.senses])

## 2단계. Gemini로 문맥 판별 + 정확도 평가
- **baseline**: 항상 첫 번째 의미(짧은 소리)를 고른다. 장단을 모르는 일반 TTS와 같은 동작.
- **gemini**: 문장 전체를 보고 의미를 고른다. 응답은 `outputs/gemini_cache.json`에 저장된다.

In [ ]:
from disambiguate import build_prompt, evaluate
print(build_prompt(s, find_candidates(s)))   # Gemini에게 실제로 보내는 질문

In [ ]:
!python run_experiment.py --step disambig
display(Markdown(open("outputs/disambiguation.md", encoding="utf-8").read()))

## 3단계-① 방법 1: 상용 TTS의 입력 바꾸기

In [ ]:
!python run_experiment.py --step commercial
for i in range(4):
    display(Image(f"outputs/commercial/{i}_mel.png"))
    display(Audio(f"outputs/commercial/{i}_base.wav")); display(Audio(f"outputs/commercial/{i}_mod.wav"))

## 3단계-② 방법 2: VITS 모델의 duration 텐서에 직접 개입
`model.duration_predictor`의 출력(log_duration, 모양 `(1, 1, 토큰 수)`)에 forward hook을 건다.
장음 모음 토큰 위치에만 `log(w)`를 더하면 `exp`를 거친 뒤 그 토큰의 프레임 수만 `w`배가 된다.

In [ ]:
from approach2_vits import VowelLengthVITS
from disambiguate import disambiguate
tts = VowelLengthVITS()
text = "눈에 눈이 들어가서 눈물이 났다."
cands = disambiguate(text)
base = tts.synthesize(text, cands, weight=1.0)
mod  = tts.synthesize(text, cands, weight=1.8)
print("로마자 입력:", mod["roman"])
print("개입한 토큰 위치:", mod["targets"])
tok = tts.tokenizer.convert_ids_to_tokens(tts.token_layout(text)[2][0])
for k in range(len(tok)):
    mark = "  ← 장음" if k in mod["targets"] else ""
    print(f"{k:3d} {tok[k]!r:6} {base['frames'][k]:3d} → {mod['frames'][k]:3d} 프레임{mark}")

In [ ]:
!python run_experiment.py --step vits --weight 1.8
display(Markdown(open("outputs/vits_result.md", encoding="utf-8").read()))
for i in range(4):
    display(Image(f"outputs/vits/{i}_mel.png"))
    display(Audio(f"outputs/vits/{i}_base.wav")); display(Audio(f"outputs/vits/{i}_mod.wav"))

## 4단계. 가중치 탐색: 몇 배가 가장 자연스러운가?
직접 들어 보고 `docs/탐구노트.md`에 느낀 점을 기록하자.

In [ ]:
for w in [1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]:
    print("가중치", w); display(Audio(f"outputs/vits/sweep_w{w}.wav"))